In [1]:
import oracledb
dsn = oracledb.makedsn("localhost", 1522, service_name="stu")
connection = oracledb.connect(user="ora_jthomp20", password="a84340637", dsn=dsn)

query1 = """
WITH QuartileData AS (
    SELECT 
        danceability, energy, tempo, loudness, acousticness, speechiness,
        NTILE(4) OVER (ORDER BY weeks_on_chart DESC) as quartile
    FROM TOP_SONG_2022
)
SELECT 
    CASE 
        WHEN quartile = 1 THEN 'Top 25%' 
        WHEN quartile = 4 THEN 'Bottom 25%' 
    END as category,
    AVG(danceability) as avg_dance,
    AVG(energy) as avg_energy,
    AVG(tempo) as avg_tempo,
    AVG(loudness) as avg_loud,
    AVG(acousticness) as avg_acoustic,
    AVG(speechiness) as avg_speech
FROM QuartileData
WHERE quartile IN (1, 4)
GROUP BY quartile
ORDER BY quartile
"""
cur = connection.cursor()
for row in cur.execute(query1):
    print(row)
cur.close()

ModuleNotFoundError: No module named 'oracledb'

In [2]:
query2 = """
SELECT 
    2022 AS year,
    SUM(CASE WHEN decile = 1 THEN weeks_on_chart ELSE 0 END) / SUM(weeks_on_chart) * 100 AS top_10_share,
    SUM(CASE WHEN quartile = 1 THEN weeks_on_chart ELSE 0 END) / SUM(weeks_on_chart) * 100 AS top_25_share
FROM (
    SELECT weeks_on_chart, 
           NTILE(10) OVER (ORDER BY weeks_on_chart DESC) AS decile,
           NTILE(4) OVER (ORDER BY weeks_on_chart DESC) AS quartile
    FROM TOP_SONG_2022
)

UNION ALL
SELECT 
    2023 AS year,
    SUM(CASE WHEN decile = 1 THEN streams ELSE 0 END) / SUM(streams) * 100 AS top_10_share,
    SUM(CASE WHEN quartile = 1 THEN streams ELSE 0 END) / SUM(streams) * 100 AS top_25_share
FROM (
    SELECT streams, 
           NTILE(10) OVER (ORDER BY streams DESC) AS decile,
           NTILE(4) OVER (ORDER BY streams DESC) AS quartile
    FROM TOP_SONG_2023
)

UNION ALL
SELECT 
    2024 AS year,
    SUM(CASE WHEN decile = 1 THEN spotify_streams ELSE 0 END) / SUM(spotify_streams) * 100 AS top_10_share,
    SUM(CASE WHEN quartile = 1 THEN spotify_streams ELSE 0 END) / SUM(spotify_streams) * 100 AS top_25_share
FROM (
    SELECT spotify_streams, 
           NTILE(10) OVER (ORDER BY spotify_streams DESC) AS decile,
           NTILE(4) OVER (ORDER BY spotify_streams DESC) AS quartile
    FROM TOP_SONG_2024
)
"""

In [3]:
cur = connection.cursor()
for row in cur.execute(query2):
    print(row)
cur.close()

NameError: name 'connection' is not defined

In [9]:
import pandas as pd
import altair as alt
df1 = pd.read_sql(query1, connection)
df2 = pd.read_sql(query2, connection)

/tmp/ipykernel_5469/3770725684.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query1, connection)
/tmp/ipykernel_5469/3770725684.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(query2, connection)


In [136]:
df1_melted = df1.melt(
    id_vars=['CATEGORY'], 
    var_name='metric', 
    value_name='value'
)
df1_melted["metric"] = df1_melted["metric"].replace({"AVG_DANCE":"Danceability","AVG_ENERGY":"Energy","AVG_TEMPO":"Tempo (BPM)","AVG_ACOUSTIC":"Acousticness","AVG_SPEECH":"Speechiness","AVG_LOUD":"Loudness"})

In [161]:
q1plot = alt.Chart(df1_melted).mark_bar().encode(
    x= alt.X("CATEGORY",title = None),
    y = alt.Y("sum(value)", title= "Value"),
    color = alt.Color("CATEGORY",legend=alt.Legend(titleFontSize=16, labelFontSize=18, symbolSize=1000,title = "Quartile")),
    facet=alt.Facet('metric:N', columns = 3, title = None)
).properties(width = 600, height = 600, title = "Mean Descriptive Musical Statistics of Top and Bottom Quartiles in 2022").resolve_scale(
    y='independent'
).configure_axisY(
    titleFontSize=22,
    labelFontSize=16).configure_header(
    labelFontSize=20, 
    labelPadding=10,     
    titleFontSize=24).configure_title(
    fontSize=24,
    anchor='middle',
    offset=20).configure_axisX(
    titleFontSize=22,
    labelFontSize=20)
q1plot

alt.Chart(...)

In [85]:
df2_melted = df2.melt(id_vars='YEAR', 
                    value_vars=['TOP_10_SHARE', 'TOP_25_SHARE'], 
                    var_name='Quantile', 
                    value_name='Share')
df2_melted["Quantile"] = df2_melted["Quantile"].replace({"TOP_10_SHARE":"90th Percentile","TOP_25_SHARE":"75th Percentile"})

In [91]:
q2plot = alt.Chart(df2_melted).mark_line().encode(
    x= alt.X("YEAR:O", axis=alt.Axis(labelAngle=0), title = "Year"),
    y = alt.Y("Share", title = "Share of Popularity (%)"),
    color = alt.Color("Quantile", title = "Percentile")
).properties(width = 600,height = 600, title = "Share of Popularity of the Top Quantiles Of Spotify Music Over Time")

In [92]:
q2plot

alt.Chart(...)

In [54]:
q2_plot = (top_10 + top_25).properties(width = 800, height = 800,title = "Share of Popularity of the Top Quantiles Of ")

q2_plot

alt.LayerChart(...)